# Timeline / Track Demo

Notebook de test pour `Timeline`, `Track` et `Keyframe` avec plusieurs types d'interpolation.
Chaque variable est tracee dans un subplot horizontal, et les keyframes sont affichees en points rouges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pixelprism.anim.timeline as timeline_mod
from pixelprism.anim.timeline import Timeline, Keyframe
from pixelprism.anim.easing import EASINGS, INTERPOLATES, Easings, Interpolates

In [ ]:
# Compat layer locale au notebook (ne modifie pas la librairie).
def apply_timeline_compatibility_patches() -> None:
    timeline_mod.Easings = Easings()
    timeline_mod.Interpolates = Interpolates()

    timeline_mod.Segment.__dataclass_fields__["easing"].default = EASINGS["linear"]
    timeline_mod.Segment.__dataclass_fields__["interpolation"].default = INTERPOLATES["interpolate"]
    timeline_mod.Segment.easing = EASINGS["linear"]
    timeline_mod.Segment.interpolation = INTERPOLATES["interpolate"]

    def _previous_keyframe(self, t: float):
        candidates = [k for k in self._keyframes if k.time < t]
        return max(candidates, key=lambda k: k.time) if candidates else None

    def _next_keyframe(self, t: float):
        candidates = [k for k in self._keyframes if k.time > t]
        return min(candidates, key=lambda k: k.time) if candidates else None

    timeline_mod.Track._previous_keyframe = _previous_keyframe
    timeline_mod.Track._next_keyframe = _next_keyframe


apply_timeline_compatibility_patches()

In [ ]:
timeline = Timeline()

# x: interpolation float + easing parametre
x = timeline.track("x", 0.0)
x.add_keyframe(Keyframe(1.0, 100.0))
x.add_keyframe(Keyframe(2.0, 40.0))
x.add_keyframe(Keyframe(3.0, 120.0))
x.set_interpolation(0.5, EASINGS["ease_in_quad"], INTERPOLATES["interpolate"])
x.set_interpolation(1.5, lambda t: EASINGS["ease_out_back"](t, overshoot=2.8), INTERPOLATES["interpolate"])
x.set_interpolation(2.5, EASINGS["ease_in_out_sine"], INTERPOLATES["interpolate"])

# radius: interpolation entiere
radius = timeline.track("radius", 5)
radius.add_keyframe(Keyframe(1.0, 12))
radius.add_keyframe(Keyframe(2.0, 4))
radius.add_keyframe(Keyframe(3.0, 16))
radius.set_interpolation(0.5, EASINGS["ease_out_cubic"], INTERPOLATES["integer"])
radius.set_interpolation(1.5, EASINGS["ease_in_back"], INTERPOLATES["integer"])
radius.set_interpolation(2.5, EASINGS["ease_out_bounce"], INTERPOLATES["integer"])

# alpha: interpolation a paliers puis easing non lineaire
def interpolate_step_8(start, end, t, easing, **kwargs):
    return INTERPOLATES["step"](start, end, t, easing, steps=8)

alpha = timeline.track("alpha", 0.0)
alpha.add_keyframe(Keyframe(1.0, 1.0))
alpha.add_keyframe(Keyframe(2.0, 0.2))
alpha.add_keyframe(Keyframe(3.0, 1.0))
alpha.set_interpolation(0.5, EASINGS["linear"], interpolate_step_8)
alpha.set_interpolation(1.5, EASINGS["ease_in_out_quad"], INTERPOLATES["interpolate"])
alpha.set_interpolation(2.5, lambda t: EASINGS["ease_in_out_elastic"](t, amplitude=1.2, period=0.5), INTERPOLATES["interpolate"])

# y: track avec hold sur le 2e keyframe
y = timeline.track("y", -20.0)
y.add_keyframe(Keyframe(1.0, 50.0))
y.add_keyframe(Keyframe(2.0, 50.0, hold=True))
y.add_keyframe(Keyframe(3.0, -10.0))
y.set_interpolation(0.5, EASINGS["ease_out_quad"], INTERPOLATES["interpolate"])
y.set_interpolation(1.5, EASINGS["linear"], INTERPOLATES["interpolate"])
y.set_interpolation(2.5, EASINGS["ease_in_expo"], INTERPOLATES["interpolate"])

track_names = list(timeline._tracks.keys())
track_names

In [ ]:
end_time = max(track.duration for track in timeline._tracks.values())
time_samples = np.linspace(0.0, end_time, 600)

n_tracks = len(track_names)
fig, axes = plt.subplots(n_tracks, 1, figsize=(14, 2.8 * n_tracks), sharex=True, squeeze=False)
axes = axes[:, 0]

for idx, var_name in enumerate(track_names):
    ax = axes[idx]
    track = timeline._tracks[var_name]

    values = np.array([timeline.evaluate(float(t))[var_name] for t in time_samples], dtype=float)

    key_times = np.array([k.time for k in track.keyframes], dtype=float)
    key_values = np.array([k.value for k in track.keyframes], dtype=float)

    ax.plot(time_samples, values, color="tab:blue", linewidth=2, label=var_name)
    ax.scatter(key_times, key_values, color="red", s=35, zorder=3, label="keyframes")

    ax.set_title(var_name)
    ax.set_ylabel("value")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("t (s)")
plt.suptitle("Timeline values per track", y=1.02)
plt.tight_layout()
plt.show()